# Perplexity OCS Bug Reproduction Tests

These cells intentionally reproduce known API errors to confirm the issue in the GitHub Dimagi Open Chat Studio product integration where there is an error with Perplexity

- Sonar API (chat completions style): `https://api.perplexity.ai/chat/completions`
- Agent API (OpenAI SDK compatible, responses style): `https://api.perplexity.ai/v1`

### Sonar API via `requests` — wrong base URL (404)

Intentionally posts to `https://api.perplexity.ai/` instead of the `/chat/completions` endpoint, to reproduce the 404 the OCS integration hits.

In [ ]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("PERPLEXITY_API_KEY")

if not api_key:
    raise ValueError("PERPLEXITY_API_KEY environment variable not set")

url = "https://api.perplexity.ai/"  # this endpoint causes the 404 error
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
data = {
    "model": "sonar",
    "messages": [{"role": "user", "content": "What is day of week for 2024-06-01?"}],
    "max_tokens": 50,
}

response = requests.post(url, headers=headers, json=data)

if response.status_code == 200:
    result = response.json()
    print(f"Response ID: {result.get('id', 'N/A')}")
    print(result["choices"][0]["message"]["content"])
else:
    if response.status_code == 404:
        print(
            "Error code 404: Endpoint not found. Please check the API documentation for the correct endpoint."
        )
    else:
        print(f"Error: {response.status_code} - {response.text}")

### Agent API via OpenAI SDK — Sonar model (400)

The Agent API's `/v1` responses endpoint doesn't support Sonar models — passing `model="sonar"` here reproduces `400: model "sonar" is not supported`. Sonar requires the chat-completions endpoint instead (see the previous cell/notebook).

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()
perplex_api_key = os.getenv("PERPLEXITY_API_KEY")

client = OpenAI(api_key=perplex_api_key, base_url="https://api.perplexity.ai/v1")

try:
    response = client.responses.create(
        model="sonar",
        input="In one sentence, what were the results of the 2025 French Open Finals?",
    )
    print(response.output_text)
except Exception as e:
    print(f"API request failed ({type(e).__name__}): {e}")

### Agent API via OpenAI SDK — wrong base URL (404)

Points `base_url` at the chat-completions endpoint instead of `https://api.perplexity.ai/v1` (the correct Agent API base). Reproduces `404 NotFoundError` — the Agent API only works via `/v1`, not the chat-completions path.

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()
perplex_api_key = os.getenv("PERPLEXITY_API_KEY")

client = OpenAI(
    api_key=perplex_api_key, base_url="https://api.perplexity.ai/chat/completions"
)

try:
    response = client.responses.create(
        model="openai/gpt-5-mini",
        input="In one sentence, what were the results of the 2025 French Open Finals?",
    )
    print(response.output_text)
except Exception as e:
    print(f"API request failed ({type(e).__name__}): {e}")